# MMMU-Pro Reproduction Run

Real, executed cells from the Kaggle session that produced this project's
MMMU-Pro results (Qwen2.5-VL-7B-Instruct vs Qwen3-VL-4B-Instruct, full
256-group standard-4-option split). Outputs below are copied verbatim from
that session, not re-run for this notebook.

Scratch/debugging cells (package version fights, stale-dataset checks, etc.
-- see `docs/TROUBLESHOOTING.md` for what those were and their fixes) have
been removed; this keeps only the sequence that actually produced the
reported numbers.

## 1. Get the code and set up the session

In [2]:
# --- 1. get the code -------------------------------------------------------
# Kaggle auto-extracts an uploaded zip, and where the files land depends on how
# the zip was built: /kaggle/input/<ds>/, /kaggle/input/<ds>/ap/, or a zip left
# sitting inside the dataset. Search instead of guessing one layout.
import os, glob, shutil, sys

MARKER   = "build_matching.py"
CODE     = "/kaggle/working/answerpool"
# Set this if the search below misses your dataset. It is the folder that
# directly contains build_matching.py, e.g.
#   "/kaggle/input/datasets/<user>/<slug>/ap"
CODE_SRC = "/kaggle/input/datasets/nawafbarbood/rrrrrrrrrrrrr/ap"

def _find(root, depth=8):
    hits = []
    if not os.path.isdir(root):
        return hits
    for d, dirs, files in os.walk(root):
        if d[len(root):].count(os.sep) > depth:
            dirs[:] = []
            continue
        dirs[:] = [x for x in dirs if not x.startswith(".")]
        if MARKER in files:
            hits.append(d)
    return sorted(hits, key=len)

src = CODE_SRC or next(iter(_find("/kaggle/input")), None)
if src and not os.path.exists(os.path.join(src, MARKER)):
    raise SystemExit(f"CODE_SRC={src!r} does not contain {MARKER}")
if src is None:                      # a .zip inside the dataset also works
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        shutil.rmtree("/kaggle/working/code_tmp", ignore_errors=True)
        shutil.unpack_archive(z, "/kaggle/working/code_tmp")
        src = next(iter(_find("/kaggle/working/code_tmp")), None)
        if src:
            print("unpacked", z)
            break

if src is None:
    print(sorted(glob.glob("/kaggle/input/*")) or "  (nothing attached)")
    raise SystemExit("\n".join([
        "Could not find the code.",
        "Add it as a Kaggle Dataset:",
        "  1. kaggle.com/datasets -> New Dataset -> upload AnswerPool_benchmarks.zip",
        "  2. in this notebook: Input -> Add Input -> your dataset",
        "  3. re-run this cell",
        "A private GitHub repo cannot be cloned here: git has no credentials and",
        "prompts for a username, which is the error you saw. If the repo is private,",
        "either upload the zip as above, or add a token secret and clone",
        "https://<TOKEN>@github.com/<user>/<repo>."]))

os.makedirs(CODE, exist_ok=True)
for f in glob.glob(src + "/*.py") + glob.glob(src + "/*.sh") + glob.glob(src + "/*.txt")         + glob.glob(src + "/*.md"):
    shutil.copy(f, CODE)
os.chdir(CODE)
sys.path.insert(0, CODE)
print("code from:", src)
print(sorted(os.listdir(CODE)))

# Sanity check the fixes below actually landed -- a stale dataset upload is
# the single most common cause of "I already fixed this" showing up again.
checks = {
    "build_matching.py": "min-options",
    "run_matching.py": "patch_rope_conflict",
    "models.py": "gemini-3.6-flash",
}
print("")
print("fix check (all should say YES):")
for fname, needle in checks.items():
    body = open(fname.strip(), encoding="utf-8").read() if os.path.exists(fname.strip()) else ""
    ok = "YES" if needle in body else "NO -- reupload the zip"
    print("  " + fname.ljust(22) + " has " + repr(needle) + ": " + ok)

code from: /kaggle/input/datasets/nawafbarbood/rrrrrrrrrrrrr/ap
['README.md', '__pycache__', 'bench_datasets.py', 'build_matching.py', 'check_data.py', 'check_fit.py', 'compare.py', 'consolidate_report.py', 'diagnose.py', 'estimate_filter.py', 'filter_blind.py', 'judge_freeform.py', 'kaggle_answerpool.py', 'mmmu_pro_co.jsonl', 'mmmu_pro_flagged.txt', 'mmmu_pro_match.jsonl', 'mmmu_pro_match0.jsonl', 'mmmu_pro_match_images', 'mmmu_pro_mcq.jsonl', 'mmmu_pro_noimg.jsonl', 'mmmu_pro_wellformed.jsonl', 'mmmu_pro_wellformed.jsonl.partial.jsonl', 'mmmu_pro_wh.jsonl', 'mmmu_pro_wh_images', 'models.py', 'own_set.py', 'position_bias.py', 'probe_api.py', 'report_tables.py', 'requirements.txt', 'rescore.py', 'run_benchmarks.sh', 'run_matching.py', 'run_report.sh', 'smoke_test.py', 'test_detail.jsonl', 'test_flagged.txt', 'validity.py', 'verify_wellformed.py']

fix check (all should say YES):
  build_matching.py      has 'min-options': YES
  run_matching.py        has 'patch_rope_conflict': YES
  mo

In [3]:
# --- 2. caches and secrets -------------------------------------------------
# /kaggle/working is capped at ~20 GB and is saved with the notebook; dataset
# and model caches must NOT go there or the commit will fail. /kaggle/temp is
# scratch and disappears at session end, which is what we want for caches.
os.environ["HF_HOME"] = "/kaggle/temp/hf"
# hf_transfer (the accelerated Rust downloader) has been observed to hang at
# 0% indefinitely on Kaggle's network for large repos (15GB+ vision models);
# plain HTTP is slower per-byte but actually starts and completes reliably.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
os.makedirs("/kaggle/temp/hf", exist_ok=True)

OUT = "/kaggle/working/runs"        # results + images: these DO persist
os.makedirs(OUT, exist_ok=True)

WANT = ["HF_TOKEN", "ANTHROPIC_API_KEY", "OPENAI_API_KEY", "MOONSHOT_API_KEY",
        "KIMI_API_KEY", "OPENROUTER_API_KEY", "GOOGLE_API_KEY", "OPENAI_BASE_URL"]
found, missing = [], []
try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    for name in WANT:
        try:
            v = sec.get_secret(name)
            if v:
                os.environ[name] = v
                found.append(name)
            else:
                missing.append(name)
        except Exception:
            missing.append(name)
except ImportError:
    print("not on Kaggle; set the environment variables yourself")

print("secrets loaded :", ", ".join(found) or "NONE")
print("not available  :", ", ".join(missing))
if not found:
    # A silent failure here is what makes cell 8 die with "set <KEY>" much
    # later, after the datasets are downloaded and the arms are built.
    for line in [
            "",
            "!! No secrets reached this kernel. Two steps are needed, and the",
            "   second is the one people miss:",
            "     1. Add-ons -> Secrets -> Add secret (label = ANTHROPIC_API_KEY)",
            "     2. tick the ATTACH checkbox for THIS notebook in that same",
            "        dialog, then re-run this cell",
            "   Without step 2 the secret exists on your account but the kernel",
            "   cannot read it."]:
        print(line)

!df -h /kaggle/working /kaggle/temp | cat

secrets loaded : HF_TOKEN, GOOGLE_API_KEY
not available  : ANTHROPIC_API_KEY, OPENAI_API_KEY, MOONSHOT_API_KEY, KIMI_API_KEY, OPENROUTER_API_KEY, OPENAI_BASE_URL
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  1.1G   19G   6% /kaggle/working
overlay         7.9T  6.9T 1019G  88% /


## 2. Confirm the working environment

(`transformers==4.57.6`, `huggingface-hub<1.0`, and vLLM's own resolved `torch` build -- the one combination that worked cleanly after several dependency conflicts; see `docs/TROUBLESHOOTING.md`.)

In [1]:
import os; print(os.getcwd())  # must print /kaggle/working

/kaggle/working


In [4]:
import torch, vllm, transformers
print(torch.__version__, vllm.__version__, transformers.__version__)

2.8.0+cu128 0.11.0 4.57.6


## 3. Build the four arms (N=5, M=15-20, min-options=4, withhold=0.4)

In [5]:
!python build_matching.py --dataset mmmu_pro --mmmu-pro-config "standard (4 options)" --n 5 --distractors --min-options 4 --withhold 0.4 --out mmmu_pro_match.jsonl
!python build_matching.py --dataset mmmu_pro --min-options 4 --arm mcq --mirror mmmu_pro_match.jsonl --out mmmu_pro_mcq.jsonl
!python build_matching.py --dataset mmmu_pro --mmmu-pro-config "standard (4 options)" --n 5 --distractors --min-options 4 --arm choices_only --out mmmu_pro_co.jsonl
!python build_matching.py --dataset mmmu_pro --min-options 4 --arm mcq_no_passage --mirror mmmu_pro_match.jsonl --out mmmu_pro_noimg.jsonl
!ls -la mmmu_pro_*.jsonl

loaded 1664 MMMU-Pro items (standard (4 options)), 66 dropped, grouping by subject
answer index range: [0, 6]   options per item: [2, 3, 4, 5, 6, 7, 8, 9]
--min-options 4: dropped 290 items (17.4%) -> 1374 remain
30 topics

wrote 256 groups -> mmmu_pro_match.jsonl   (skipped 8 for duplicate golds)
padding: 0 cross-article candidates total (GOOD - all same-article)
pool size M varies: min 12, median 18, max 18 (option counts differ per item)
floors at N=5: per-pair 1/M in [0.0556, 0.0833], worst-case exact-assignment 1/95,040 = 1.05e-05
  -> quote the worst-case floor, or use --min-options to make M constant
images: 1342 written under mmmu_pro_match_images/ (paths stored in each record's 'images'; run_matching.py attaches them)

--- sample prompt (truncated) ---
Match each question to its correct answer from the candidate list.

Each candidate may be used at most once.
If a question's answer is NOT among the candidates, write "none" for it.
Images referenced by a question are attached t

## 4. Run Qwen2.5-VL-7B-Instruct (full 256 groups)

In [6]:
!python run_matching.py --in mmmu_pro_match.jsonl --model Qwen/Qwen2.5-VL-7B-Instruct --backend vllm --limit 0 --tp 2 --max-len 32768
!python run_matching.py --in mmmu_pro_mcq.jsonl --model Qwen/Qwen2.5-VL-7B-Instruct --backend vllm --limit 0 --tp 2 --max-len 32768
!python run_matching.py --in mmmu_pro_co.jsonl --model Qwen/Qwen2.5-VL-7B-Instruct --backend vllm --limit 0 --tp 2 --max-len 32768
!python run_matching.py --in mmmu_pro_wh.jsonl --model Qwen/Qwen2.5-VL-7B-Instruct --backend vllm --limit 0 --tp 2 --max-len 32768
!python run_matching.py --in mmmu_pro_noimg.jsonl --model Qwen/Qwen2.5-VL-7B-Instruct --backend vllm --limit 0 --tp 2 --max-len 8192

arm=matching N=5 M=18 | 256 records | model=Qwen/Qwen2.5-VL-7B-Instruct | backend=vllm
INFO 09-16 08:03:20 [__init__.py:216] Automatically detected platform cuda.
config.json: 1.37kB [00:00, 3.61MB/s]
[rope-patch] config.rope_scaling: dropping legacy type='default', keeping rope_type='default'
[rope-patch] patched 1 conflict(s), saved to cache at /kaggle/temp/hf/hub/models--Qwen--Qwen2.5-VL-7B-Instruct/snapshots/cc594898137f460bfe9f0759e9844b3ce807cfb5
tokenizer_config.json: 5.70kB [00:00, 29.6MB/s]
vocab.json: 2.78MB [00:00, 15.0MB/s]
merges.txt: 1.67MB [00:00, 15.5MB/s]
tokenizer.json: 7.03MB [00:00, 23.2MB/s]
[vllm] Tesla T4 (sm_75) has no bfloat16 -> dtype=float16
INFO 09-16 08:03:42 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 32768, 'tensor_parallel_size': 2, 'disable_log_stats': True, 'limit_mm_per_prompt': {'image': 9}, 'model': 'Qwen/Qwen2.5-VL-7B-Instruct'}
The argument `trust_remote_code` is to be used with Auto classes. I

## 5. Run Qwen3-VL-4B-Instruct (full 256 groups)

`--enforce-eager` is required here: without it, vLLM's CUDA-graph capture step OOMs on a T4 even though the model itself fits comfortably (see `docs/TROUBLESHOOTING.md`).

In [7]:
!python run_matching.py --in mmmu_pro_match.jsonl --model Qwen/Qwen3-VL-4B-Instruct --backend vllm --limit 0 --tp 2 --max-len 32768 --enforce-eager
!python run_matching.py --in mmmu_pro_mcq.jsonl --model Qwen/Qwen3-VL-4B-Instruct --backend vllm --limit 0 --tp 2 --max-len 32768 --enforce-eager
!python run_matching.py --in mmmu_pro_co.jsonl --model Qwen/Qwen3-VL-4B-Instruct --backend vllm --limit 0 --tp 2 --max-len 32768 --enforce-eager
!python run_matching.py --in mmmu_pro_wh.jsonl --model Qwen/Qwen3-VL-4B-Instruct --backend vllm --limit 0 --tp 2 --max-len 32768 --enforce-eager
!python run_matching.py --in mmmu_pro_noimg.jsonl --model Qwen/Qwen3-VL-4B-Instruct --backend vllm --limit 0 --tp 2 --max-len 8192 --enforce-eager

arm=matching N=5 M=18 | 256 records | model=Qwen/Qwen3-VL-4B-Instruct | backend=vllm
INFO 09-16 11:11:13 [__init__.py:216] Automatically detected platform cuda.
config.json: 1.50kB [00:00, 672kB/s]
tokenizer_config.json: 10.9kB [00:00, 50.8MB/s]
vocab.json: 2.78MB [00:00, 146MB/s]
merges.txt: 1.67MB [00:00, 153MB/s]
tokenizer.json: 7.03MB [00:00, 199MB/s]
[vllm] Tesla T4 (sm_75) has no bfloat16 -> dtype=float16
INFO 09-16 11:11:23 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 32768, 'tensor_parallel_size': 2, 'disable_log_stats': True, 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 9}, 'model': 'Qwen/Qwen3-VL-4B-Instruct'}
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
preprocessor_config.json: 100%|████████████████| 390/390 [00:00<00:00, 2.88MB/s]
INFO 09-16 11:11:43 [model.py:547] Resolved architecture: Qwen3VLForConditionalGeneration
`torch_dtype` is deprecated! Use 

## 6. Verifier screen (Gemini, 2 screeners, votes=2 majority)

Checks whether any candidate answer is ALSO a correct answer to another question in the same pool -- the paper's Section 3.3 well-formedness check, extended here to attach each question's own image.

In [15]:
!python verify_wellformed.py --src mmmu_pro_match0.jsonl --out mmmu_pro_flagged_v2.txt --detail mmmu_pro_wellformed_v2.jsonl --models "gemini-3.6-flash,gemini-3.1-flash-lite" --votes 2

256 groups x 5 questions x 2 screeners = 2560 calls
[client] Gemini API key
[image_parts] SKIPPING unreadable image mmmu_pro_match_images/1a6b09bdf9bbd73f_1.png (OSError: image file is truncated); the model will judge this question with the image missing, which is safer than aborting the whole batch. Rebuild the .jsonl if this question's answer genuinely needs that image.
[image_parts] SKIPPING unreadable image mmmu_pro_match_images/1a6b09bdf9bbd73f_1.png (OSError: image file is truncated); the model will judge this question with the image missing, which is safer than aborting the whole batch. Rebuild the .jsonl if this question's answer genuinely needs that image.
  2560/2560 new calls done

screeners disagree-with-uniqueness on 12/1280 questions (0.9%)
  named candidate is another question's gold : 0  (W1/W2)
  named candidate is a distractor            : 12  (cross-valid)
flagged groups: 11/256  ->  RETENTION 95.7%
unparsed screener replies 60, api errors 0 (both counted as clean, r

## 7. Cross-model comparison

This was corrected in a follow-up cell after an initial run mistakenly
excluded `mmmu_pro_flagged.txt` (20 groups, an earlier single-screener
pass) instead of `mmmu_pro_flagged_v2.txt` (11 groups, the `--votes 2`
two-screener majority result from cell 6 above, which is the intended and
reported check). The cell below uses the corrected exclusion file.

In [17]:
!python compare.py --glob "mmmu_pro_*.results.jsonl" --exclude-groups mmmu_pro_flagged_v2.txt

excluding 11 flagged groups from every table

model                                  choices_only+d    matching+d+wh40 matching+d+wh40:mmmu_pro_wh                mcq     mcq_no_passage
---------------------------------------------------------------------------------------------------------------------------------
5-VL-7B-Instruct                      0.0410 (n=1220)     0.3184 (n=735)     0.3184 (n=735)    0.4629 (n=1225)    0.3429 (n=1225)
Qwen_Qwen3-VL-4B-Instruct             0.0443 (n=1220)     0.3306 (n=735)     0.3306 (n=735)    0.4849 (n=1225)    0.3567 (n=1225)

EXACT ASSIGNMENT (every slot in the group correct; unparsed = wrong)
model                                  choices_only+d    matching+d+wh40 matching+d+wh40:mmmu_pro_wh                mcq     mcq_no_passage
5-VL-7B-Instruct                       0.0000 (k=244)     0.0490 (k=245)     0.0490 (k=245)    0.4629 (k=1225)    0.3429 (k=1225)
Qwen_Qwen3-VL-4B-Instruct              0.0000 (k=244)     0.0653 (k=245)     0.0653 (k

## 8. Data-quality check: corrupted images on disk

One PNG (of 1,342) was truncated, likely from an interrupted earlier build. `image_parts()` in `run_matching.py` skips unreadable images with a warning rather than crashing the run; this cell confirms the damage was limited to a single file.

In [9]:
import os
from PIL import Image

bad = []
for f in os.listdir("mmmu_pro_match_images"):
    path = os.path.join("mmmu_pro_match_images", f)
    try:
        Image.open(path).load()
    except Exception as e:
        bad.append((path, str(e)))

print(f"corrupted images: {len(bad)} / {len(os.listdir('mmmu_pro_match_images'))}")
for p, e in bad:
    print(" ", p, "-", e)

corrupted images: 1 / 1342
  mmmu_pro_match_images/1a6b09bdf9bbd73f_1.png - image file is truncated
